## Overview
This notebook implements the data splitting pipeline for prior regulatory knowledge. Inputs:
TF.csv: List of Transcription Factors (TFs).
Target.csv: List of Target genes.
label.csv: The gold-standard dataset of known TF-Target regulatory pairs.
We partition the gold-standard regulatory pairs into strictly non-overlapping training and test sets. Furthermore, 20% of the training set is reserved for validation.

### 1. Data display

In [36]:
import os
print("current working directory:", os.getcwd())
os.chdir('/home/common/llliu/nar/grn_project/')
print("The current working directory has been changed to:", os.getcwd())

current working directory: /home/common/llliu/nar/grn_project
The current working directory has been changed to: /home/common/llliu/nar/grn_project


In [12]:
import pandas as pd
import numpy as np
TF2file = './processed_data/' +'mHSC-E' + '/TF.csv'
Gene2file = './processed_data/' + "mHSC-E" + '/Target.csv'
label_file = 'processed_data/' + "mHSC-E" + '/label.csv'

gene_set = pd.read_csv(Gene2file)['index'].values
tf_set = pd.read_csv(TF2file)['index'].values
label = pd.read_csv(label_file)
label_list = label.values.tolist()
print("TF:{}".format(tf_set))
print("Target:{}".format(gene_set))
print("true label:{}".format(label))

TF:[ 577  100  712  919  251  962  715  840  692  356 1143  769   64  295
  716  778  184    9  144  944  421  986  843  146  834 1162  355   20
 1006  804  351 1064    2]
Target:[   0    1    2 ... 1201 1202 1203]
true label:        TF  Target
0      577     476
1      577    1143
2      577     919
3      577     959
4      577     356
...    ...     ...
21970    2     131
21971    2     622
21972    2     399
21973    2      51
21974    2     108

[21975 rows x 2 columns]


### 2. Data splitting

In [24]:

tf = label['TF'].values
tf_list = np.unique(tf)
pos_dict = {}
for i in tf_list:
    pos_dict[i] = []
for i, j in label.values:
    pos_dict[i].append(j)


train_pos = {}
val_pos = {}
test_pos = {}
for k in pos_dict.keys():
    if len(pos_dict[k]) <= 1:
        p = np.random.uniform(0, 1)
        if p <= 0.5:
            train_pos[k] = pos_dict[k]
        else:
            test_pos[k] = pos_dict[k]
    elif len(pos_dict[k]) == 2:
        train_pos[k] = [pos_dict[k][0]]
        test_pos[k] = [pos_dict[k][1]]
    else:
        np.random.shuffle(pos_dict[k])
        train_pos[k] = pos_dict[k][:len(pos_dict[k]) * 2 // 3]
        test_pos[k] = pos_dict[k][len(pos_dict[k]) * 2 // 3:]
        val_pos[k] = train_pos[k][:len(train_pos[k]) // 5]
        train_pos[k] = train_pos[k][len(train_pos[k]) // 5:]

#print("train_pos:{}".format(train_pos))
#print("val_pos:{}".format(val_pos))
#print("test_pos:{}".format(test_pos))

total_pairs1 = 0
for tf_id, target_list in train_pos.items():
    tf_pairs1 = 1 * len(target_list)
    total_pairs1 += tf_pairs1
    print(f"TF {tf_id}: corresponds to {len(target_list)} Targets, number of regulatory pairs = {tf_pairs}")

total_pairs2 = 0
for tf_id, target_list in val_pos.items():
    tf_pairs2 = 1 * len(target_list)
    total_pairs2 += tf_pairs2
    #print(f"TF {tf_id}: corresponds to {len(target_list)} Targets, number of regulatory pairs = {tf_pairs}")

total_pairs3 = 0
for tf_id, target_list in test_pos.items():
    tf_pairs3 = 1 * len(target_list)
    total_pairs3 += tf_pairs3
    #print(f"TF {tf_id}: corresponds to {len(target_list)} Targets, number of regulatory pairs = {tf_pairs}")

print(f"\nTotal number of all train TF-Target regulatory pairs: {total_pairs1}")
print(f"\nTotal number of all val TF-Target regulatory pairs: {total_pairs2}")
print(f"\nTotal number of all test TF-Target regulatory pairs: {total_pairs3}")

TF 2: corresponds to 496 Targets, number of regulatory pairs = 612
TF 9: corresponds to 573 Targets, number of regulatory pairs = 612
TF 20: corresponds to 12 Targets, number of regulatory pairs = 612
TF 64: corresponds to 541 Targets, number of regulatory pairs = 612
TF 100: corresponds to 570 Targets, number of regulatory pairs = 612
TF 144: corresponds to 304 Targets, number of regulatory pairs = 612
TF 146: corresponds to 328 Targets, number of regulatory pairs = 612
TF 184: corresponds to 205 Targets, number of regulatory pairs = 612
TF 251: corresponds to 404 Targets, number of regulatory pairs = 612
TF 295: corresponds to 602 Targets, number of regulatory pairs = 612
TF 351: corresponds to 616 Targets, number of regulatory pairs = 612
TF 355: corresponds to 11 Targets, number of regulatory pairs = 612
TF 356: corresponds to 521 Targets, number of regulatory pairs = 612
TF 421: corresponds to 491 Targets, number of regulatory pairs = 612
TF 577: corresponds to 36 Targets, number 

### 3. Overlap check

In [26]:
def dict_to_pairs_set(input_dict):
    pairs_set = set()
    for tf_id, target_list in input_dict.items():
        for target in target_list:
            pairs_set.add((tf_id, target))
    return pairs_set

train_pairs = dict_to_pairs_set(train_pos)
val_pairs = dict_to_pairs_set(val_pos)
test_pairs = dict_to_pairs_set(test_pos)

train_val_overlap = train_pairs & val_pairs
train_test_overlap = train_pairs & test_pairs
val_test_overlap = val_pairs & test_pairs

print("=" * 60)
print(f"Total TF-Target pairs in training set: {len(train_pairs)}")
print(f"Total TF-Target pairs in validation set: {len(val_pairs)}")
print(f"Total TF-Target pairs in test set: {len(test_pairs)}")
print("=" * 60)

print(f"\n1. Overlapping pairs between Training and Validation sets: {len(train_val_overlap)}")
if train_val_overlap:
    print(f"   Overlapping pairs: {sorted(train_val_overlap)}")

print(f"\n2. Overlapping pairs between Training and Test sets: {len(train_test_overlap)}")
if train_test_overlap:
    print(f"   Overlapping pairs: {sorted(train_test_overlap)}")

print(f"\n3. Overlapping pairs between Validation and Test sets: {len(val_test_overlap)}")
if val_test_overlap:
    print(f"   Overlapping pairs: {sorted(val_test_overlap)}")

total_overlap = len(train_val_overlap) + len(train_test_overlap) + len(val_test_overlap)
print(f"\nTotal number of overlapping pairs across all sets: {total_overlap}")
if total_overlap == 0:
    print("No overlapping TF-Target regulatory pairs found between any sets.")

Total TF-Target pairs in training set: 11725
Total TF-Target pairs in validation set: 2913
Total TF-Target pairs in test set: 7337

1. Overlapping pairs between Training and Validation sets: 0

2. Overlapping pairs between Training and Test sets: 0

3. Overlapping pairs between Validation and Test sets: 0

Total number of overlapping pairs across all sets: 0
No overlapping TF-Target regulatory pairs found between any sets.


### 4. Generate 'hard' negative samples

In [35]:
import numpy as np
import pandas as pd

# Training set negative sampling
train_neg = {}
MAX_TRAIN_ATTEMPTS = 100000
for k in train_pos.keys():
    train_neg[k] = []
    target_neg_num = len(train_pos[k])
    if target_neg_num == 0:
        print(f"TF{k} training negative sampling completed")
        continue

    valid_neg_pool = []
    for neg in gene_set:
        if neg != k and neg not in pos_dict[k]:
            valid_neg_pool.append(neg)

    if not valid_neg_pool:
        train_neg[k] = []
    else:
        np.random.shuffle(valid_neg_pool)
        sample_num = min(target_neg_num, len(valid_neg_pool))
        train_neg[k] = valid_neg_pool[:sample_num]

        total_attempts = 0
        while len(train_neg[k]) < target_neg_num and total_attempts < MAX_TRAIN_ATTEMPTS:
            neg = np.random.choice(gene_set)
            total_attempts += 1
            if neg != k and neg not in pos_dict[k] and neg not in train_neg[k]:
                train_neg[k].append(neg)

    print(f"TF{k} training negative sampling completed")

train_pos_set = []
train_neg_set = []
for k in train_pos.keys():
    for j in train_pos[k]:
        train_pos_set.append([k, j])
tran_pos_label = [1 for _ in range(len(train_pos_set))]

for k in train_neg.keys():
    for j in train_neg[k]:
        train_neg_set.append([k, j])
tran_neg_label = [0 for _ in range(len(train_neg_set))]

train_set = train_pos_set + train_neg_set
train_label = tran_pos_label + tran_neg_label

train_sample = []
for idx, (tf_val, target_val) in enumerate(train_set):
    train_sample.append([tf_val, target_val, train_label[idx]])
train = pd.DataFrame(train_sample, columns=['TF', 'Target', 'Label'])

# Validation set processing
val_pos_set = []
for k in val_pos.keys():
    for j in val_pos[k]:
        val_pos_set.append([k, j])
val_pos_label = [1 for _ in range(len(val_pos_set))]

train_edges_set = set((int(tf_val), int(target_val)) for tf_val, target_val in train_set)

val_neg = {}
count = 0
MAX_VAL_ATTEMPTS = 100000
for k in val_pos.keys():
    val_neg[k] = []
    target_neg_num = len(val_pos[k])
    if target_neg_num == 0:
        count += 1
        print(f"TF{k} validation negative sampling completed")
        continue

    valid_neg_pool = []
    for neg in gene_set:
        if (neg != k and neg not in pos_dict[k] and
                (int(k), int(neg)) not in train_edges_set and neg not in train_neg[k]):
            valid_neg_pool.append(neg)

    if not valid_neg_pool:
        val_neg[k] = []
    else:
        np.random.shuffle(valid_neg_pool)
        sample_num = min(target_neg_num, len(valid_neg_pool))
        val_neg[k] = valid_neg_pool[:sample_num]

        total_attempts = 0
        while len(val_neg[k]) < target_neg_num and total_attempts < MAX_VAL_ATTEMPTS:
            neg = np.random.choice(gene_set)
            total_attempts += 1
            if (neg != k and neg not in pos_dict[k] and
                    (int(k), int(neg)) not in train_edges_set and
                    neg not in train_neg[k] and neg not in val_neg[k]):
                val_neg[k].append(neg)

    count += 1
    print(f"TF{k} validation negative sampling completed")

val_neg_set = []
for k in val_neg.keys():
    for j in val_neg[k]:
        val_neg_set.append([k, j])

val_neg_label = [0 for _ in range(len(val_neg_set))]
val_set = val_pos_set + val_neg_set
val_set_label = val_pos_label + val_neg_label

val_set_a = np.array(val_set)
val_sample = pd.DataFrame()
val_sample['TF'] = val_set_a[:, 0]
val_sample['Target'] = val_set_a[:, 1]
val_sample['Label'] = val_set_label

# Test set processing
test_pos_set = []
for k in test_pos.keys():
    for j in test_pos[k]:
        test_pos_set.append([k, j])

count = sum(len(v) for v in test_pos.values())

train_edges = set((int(tf_val), int(target_val)) for tf_val, target_val in train_set)
val_edges = set((int(tf_val), int(target_val)) for tf_val, target_val in val_set)
test_pos_edges = set((int(tf_val), int(target_val)) for tf_val, target_val in test_pos_set)
label_edges = set((int(row['TF']), int(row['Target'])) for _, row in label.iterrows())

test_neg_set = []
target_neg_num = count
MAX_TEST_ATTEMPTS = 100000
total_attempts = 0

valid_neg_pool = []
for t1 in tf_set:
    for t2 in gene_set:
        t1_int = int(t1)
        t2_int = int(t2)
        if (t1_int != t2_int and
                (t1_int, t2_int) not in train_edges and
                (t1_int, t2_int) not in test_pos_edges and
                (t1_int, t2_int) not in val_edges and
                (t1_int, t2_int) not in label_edges):
            valid_neg_pool.append((t1_int, t2_int))

if not valid_neg_pool:
    test_neg_set = [(0, 0) for _ in range(target_neg_num)]
else:
    np.random.shuffle(valid_neg_pool)
    sample_num = min(target_neg_num, len(valid_neg_pool))
    test_neg_set = valid_neg_pool[:sample_num]

    while len(test_neg_set) < target_neg_num and total_attempts < MAX_TEST_ATTEMPTS:
        t1 = np.random.choice(tf_set)
        t2 = np.random.choice(gene_set)
        t1_int = int(t1)
        t2_int = int(t2)
        total_attempts += 1

        is_valid = (
                t1_int != t2_int and
                (t1_int, t2_int) not in train_edges and
                (t1_int, t2_int) not in test_pos_edges and
                (t1_int, t2_int) not in val_edges and
                (t1_int, t2_int) not in label_edges and
                (t1_int, t2_int) not in set(test_neg_set)
        )

        if is_valid:
            test_neg_set.append((t1_int, t2_int))

test_pos_label = [1 for _ in range(len(test_pos_set))]
test_neg_label = [0 for _ in range(len(test_neg_set))]
test_set = test_pos_set + test_neg_set
test_label = test_pos_label + test_neg_label

test_sample = []
for idx, (tf_val, target_val) in enumerate(test_set):
    test_sample.append([tf_val, target_val, test_label[idx]])
test_sample = pd.DataFrame(test_sample, columns=['TF', 'Target', 'Label'])


TF2 training negative sampling completed
TF9 training negative sampling completed
TF20 training negative sampling completed
TF64 training negative sampling completed
TF100 training negative sampling completed
TF144 training negative sampling completed
TF146 training negative sampling completed
TF184 training negative sampling completed
TF251 training negative sampling completed
TF295 training negative sampling completed
TF351 training negative sampling completed
TF355 training negative sampling completed
TF356 training negative sampling completed
TF421 training negative sampling completed
TF577 training negative sampling completed
TF692 training negative sampling completed
TF712 training negative sampling completed
TF715 training negative sampling completed
TF716 training negative sampling completed
TF769 training negative sampling completed
TF778 training negative sampling completed
TF804 training negative sampling completed
TF834 training negative sampling completed
TF840 training ne